In [ ]:
# Импортируем функцию generate из app.py
from app import generate

print("Добро пожаловать в FreeSpeech TTS")
print("Используйте generate(text, out_path, ref_audio, ref_text='', speed=1.0, nfe_step=64, cross_fade=0.15)")

In [ ]:
import numpy as np
import soundfile as sf
import os
import tempfile
from app import generate

def generate_with_progress(text, out_path, ref_audio, ref_text="",
                           speed=1.0, nfe_step=64, cross_fade=0.15,
                           split_by='\n'):
    """
    Генерирует аудио, разбивая текст на части по разделителю (по умолчанию строки).
    Печатает номер текущей части.
    Склеивает части в один файл.
    """
    parts = [p.strip() for p in text.split(split_by) if p.strip()]
    if not parts:
        parts = [text]
    
    total = len(parts)
    print(f"🔹 Всего частей для генерации: {total}")
    temp_files = []
    
    for i, part in enumerate(parts):
        print(f"🔸 Генерация части {i+1}/{total}...")
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
            temp_path = tmp.name
        generate(part, temp_path, ref_audio, ref_text, speed, nfe_step, cross_fade)
        temp_files.append(temp_path)
        print(f"✅ Часть {i+1} готова")
    
    print("🔹 Склеиваем части...")
    combined = []
    sr = None
    for fpath in temp_files:
        data, sr = sf.read(fpath)
        combined.append(data)
        os.unlink(fpath)
    
    full_audio = np.concatenate(combined)
    sf.write(out_path, full_audio, sr)
    print(f"✅ Готово! Результат сохранён как: {out_path}")

print("Функция generate_with_progress определена")

In [ ]:
# Загружаем текст из файла или вставляем вручную
try:
    with open("text.txt", "r", encoding="utf-8") as f:
        text = f.read()
    print("Текст загружен из text.txt")
except FileNotFoundError:
    text = """Кто быстрее — тот и прав,
Кто чернее — тот и жид,
Кто остался — тот и свят —.
Так и надо, так и будем жить.

Безразлично, как любить и не делать зла,
Бесполезно, как пытаться не делать зла,
Понимая постепенно себя во сне,
Наблюдая воспоминанья.

Смерть — это место для тех, кто мёртв,
Смерть — это место для тех, кто жив,
Это такое место, где любить и не делать лжи,
Это Родина — Смерть."""
    print("Используется текст, вставленный вручную")

print(f"Длина текста: {len(text)} символов")

In [ ]:
# Генерация с прогрессом
generate_with_progress(
    text=text,
    out_path="output_final.wav",
    ref_audio="Man_voice_rus_1.wav",
    ref_text="",
    speed=1.0,          # можно менять
    nfe_step=64,        # качество
    cross_fade=0.15,    # плавность склейки
    split_by='\n'       # разбивка по строкам
)

In [ ]:
# Прослушать результат (опционально)
from IPython.display import Audio
Audio("output_final.wav")